
# 🚀 Big Data Analysis with PySpark (Google Colab)
**Internship Task-1**  
Perform analysis on a large dataset using PySpark to demonstrate scalability.

---
### Steps in this Notebook
1. Install & Configure Spark in Google Colab  
2. Load a large dataset (NYC Yellow Taxi)  
3. Data Cleaning  
4. Analysis & Insights  
5. Partitioned Output (Scalability)  
6. Visualization  
7. Summary of Insights  


In [ ]:

# ✅ Step 1: Install Java (required for Spark)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

# ✅ Step 2: Download Apache Spark (v3.5.0 with Hadoop3)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz

# ✅ Step 3: Extract Spark
!tar -xzf spark-3.5.0-bin-hadoop3.tgz

# ✅ Step 4: Set Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# ✅ Step 5: Install findspark
!pip install -q findspark


In [ ]:

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Start Spark session
spark = (SparkSession.builder
         .appName("BigData_Task1")
         .config("spark.sql.shuffle.partitions", "200")
         .getOrCreate())

spark


In [ ]:

# ✅ Step 3: Download dataset (NYC Taxi Jan 2023, Parquet)
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet -O taxi.parquet

# Load into Spark DataFrame
df = spark.read.parquet("taxi.parquet")

df.printSchema()
print("Total Rows:", df.count())


In [ ]:

from pyspark.sql.functions import col

# Clean dataset: cast numeric and filter invalid trips
df = (df.withColumn("fare_amount", col("fare_amount").cast("double"))
        .withColumn("trip_distance", col("trip_distance").cast("double"))
        .filter(col("trip_distance") > 0)
        .filter(col("fare_amount") > 0))

print("Rows after cleaning:", df.count())


In [ ]:

from pyspark.sql import functions as F

# ✅ Analysis 1: Trips by Payment Type
payment_summary = (df.groupBy("payment_type")
                     .agg(F.count("*").alias("num_trips"),
                          F.avg("fare_amount").alias("avg_fare"))
                     .orderBy(F.desc("num_trips")))

payment_summary.show()


In [ ]:

# ✅ Analysis 2: Monthly Trends
df2 = (df.withColumn("year", F.year("tpep_pickup_datetime"))
        .withColumn("month", F.month("tpep_pickup_datetime")))

monthly_trend = (df2.groupBy("year","month")
                   .agg(F.count("*").alias("trips"),
                        F.avg("fare_amount").alias("avg_fare"))
                   .orderBy("year","month"))

monthly_trend.show(12)


In [ ]:

# ✅ Analysis 3: Peak Hour Trips
df3 = df2.withColumn("hour", F.hour("tpep_pickup_datetime"))

hourly_trend = (df3.groupBy("hour")
                   .agg(F.count("*").alias("trips"))
                   .orderBy("hour"))

hourly_trend.show(24)


In [ ]:

# ✅ Step 6: Save Processed Data (partitioned by year & month)
(df2.write.mode("overwrite")
    .partitionBy("year","month")
    .parquet("/content/processed_taxi"))


In [ ]:

# ✅ Step 7: Visualization on a sample
sample_pdf = df.sample(0.001).toPandas()

import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.hist(sample_pdf["trip_distance"], bins=50, color="skyblue", edgecolor="black")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Frequency")
plt.title("Distribution of Trip Distances (sample)")
plt.show()



---
## ✅ Summary of Insights
- **Payment Type Analysis**: Shows which payment method (e.g., card, cash) dominates.  
- **Monthly Trend**: Trips & average fares grouped by year/month.  
- **Hourly Trend**: Peak demand hours in a day.  
- **Partitioned Data**: Demonstrates scalability by writing Parquet files partitioned by year & month.  
- **Visualization**: Distribution of trip distances shows majority short trips.  

---
👉 You can extend this by loading **multiple months** (e.g., `yellow_tripdata_2023-*.parquet`) to analyze a full year of taxi data at scale.
